# AlphaSignal — Offline Training

Trains RF, LSTM, and PPO checkpoints for all 5 assets before first deploy.
Run this once locally, then commit the `checkpoints/` folder (or let the weekly cron handle it post-deploy).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../backend'))

from app.data.fetcher import fetch_all_historical, ASSETS
from app.data.features import compute_features, get_feature_matrix, FEATURE_COLS
from app.models.random_forest import train_rf, save_rf
from app.models.train_lstm import train_lstm, save_lstm
from app.models.rl_agent import train_ppo, save_ppo
import numpy as np

print('Assets:', ASSETS)

In [ ]:
# Step 1: fetch 2 years of daily historical data
historical = fetch_all_historical()
for t, df in historical.items():
    print(t, len(df), 'rows')

In [ ]:
# Step 2: train RF + LSTM per ticker
rf_results = {}
lstm_results = {}
feature_data = {}

for ticker in ASSETS:
    df = historical.get(ticker)
    if df is None or df.empty:
        print(f'Skipping {ticker} — no data')
        continue
    df_feat = compute_features(df)
    if df_feat.empty or len(df_feat) < 100:
        print(f'Skipping {ticker} — insufficient features ({len(df_feat)} rows)')
        continue
    feature_data[ticker] = df_feat

    X, y = get_feature_matrix(df_feat)
    rf_model, rf_acc = train_rf(X, y, ticker)
    save_rf(rf_model, ticker)
    rf_results[ticker] = rf_acc
    print(f'{ticker} RF CV accuracy: {rf_acc:.4f}')

    available = [c for c in FEATURE_COLS if c in df_feat.columns] + ['close']
    feat_array = df_feat[available].values
    lstm_model, lstm_mae = train_lstm(feat_array, ticker, epochs=30)
    save_lstm(lstm_model, ticker, {'mae': lstm_mae})
    lstm_results[ticker] = lstm_mae
    print(f'{ticker} LSTM val loss: {lstm_mae:.6f}')

In [ ]:
# Step 3: train PPO using NVDA price series + synthetic signal history
# (production pipeline builds real signals at inference time; this primes
# a reasonable starting policy so PPO isn't pure heuristic on day 1)

ticker = 'NVDA'
df_feat = feature_data.get(ticker)
if df_feat is not None and len(df_feat) > 100:
    price_series = df_feat['close'].values
    signals = [
        {
            'rf_prob_up': 0.5 + np.random.uniform(-0.2, 0.2),
            'lstm_pct': np.random.uniform(-2, 2),
            'sentiment': np.random.uniform(-0.5, 0.5),
            'rsi': row['rsi'],
        }
        for _, row in df_feat.iterrows()
    ]
    ppo_model = train_ppo(price_series, signals, total_timesteps=50000, ticker='multi')
    save_ppo(ppo_model, 'multi')
    print('PPO training complete.')
else:
    print('Insufficient NVDA data for PPO training — will use heuristic fallback.')

In [ ]:
# Summary
print('=== RF CV Accuracy ===')
for t, acc in rf_results.items():
    print(f'  {t}: {acc:.2%}')

print('\n=== LSTM Validation Loss ===')
for t, mae in lstm_results.items():
    print(f'  {t}: {mae:.6f}')

print('\nCheckpoints saved to backend/checkpoints/')
print('Ready to deploy. Run: docker build -t alphasignal-api ./backend')